In [ ]:
import numpy as np

ModuleNotFoundError: No module named 'pandas'

In [5]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
import sys
sys.path.append("../src")      # let the notebook find src/
from gait import process_sequence, joint_angle, HIP, KNEE, ANKLE

# now just use them
r = process_sequence("adl-01")

print(f"{r['seq']}: {r['n']} frames, "
      f"2D valid={np.sum(~np.isnan(r['a2d']))}, "
      f"3D valid={np.sum(~np.isnan(r['a3d']))}")

I0000 00:00:1783362748.701347 39230078 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1783362748.799886 40347663 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1783362748.862662 40347663 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


adl-01: 150 frames, 2D valid=136, 3D valid=92


In [7]:
# --- Run the pipeline over several sequences and summarize ---

def summarize(r, gate_deg=25):
    """Compute comparison metrics for one sequence's results."""
    a2d, a3d = r["a2d"], r["a3d"]
    both = ~np.isnan(a2d) & ~np.isnan(a3d)
    out = {
        "seq": r["seq"],
        "frames": r["n"],
        "rgb_valid%": 100 * np.sum(~np.isnan(a2d)) / r["n"],
        "depth_valid%": 100 * np.sum(~np.isnan(a3d)) / r["n"],
        "both": int(both.sum()),
    }
    if both.sum() >= 5:
        diff = a2d[both] - a3d[both]
        keep = np.abs(diff) <= gate_deg          # exclude extreme disagreements
        out["RMSE_all"]  = float(np.sqrt(np.mean(diff**2)))
        out["RMSE_kept"] = float(np.sqrt(np.mean(diff[keep]**2))) if keep.sum() else np.nan
        out["corr"]      = float(np.corrcoef(a2d[both], a3d[both])[0, 1])
        out["mean_diff"] = float(np.mean(diff))
    return out

# sequences to test — mix of ADLs and falls
sequences = ["adl-01", "adl-02", "adl-03", "adl-04", "adl-05", "fall-01", "fall-02"]

results, summaries = {}, []
for seq in sequences:
    print(f"processing {seq}...")
    r = process_sequence(seq)
    results[seq] = r                    # keep raw angles for later plotting
    summaries.append(summarize(r))

# tidy table
import pandas as pd
df = pd.DataFrame(summaries)
df = df.round(1)
print("\n" + df.to_string(index=False))

processing adl-01...


I0000 00:00:1783362948.001202 39230078 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1783362948.079763 40351421 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1783362948.132306 40351428 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


processing adl-02...


I0000 00:00:1783362977.767919 39230078 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1783362977.838101 40352042 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1783362977.877558 40352042 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


processing adl-03...


...

processing fall-01...


I0000 00:00:1783363120.331418 39230078 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1783363120.407508 40354410 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1783363120.451018 40354417 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


processing fall-02...


I0000 00:00:1783363148.550874 39230078 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1783363148.623888 40354928 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1783363148.665339 40354929 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


ModuleNotFoundError: No module named 'pandas'